# Baseflow separation

This notebook demonstrates the baseflow separation workflow implemented in `hydroevents`.

The workflow applies the Lyne–Hollick digital filter to separate streamflow into:

- baseflow;
- stormflow.

The filter parameter can be manually specified or automatically derived from the Master Recession Curve (MRC) analysis.

## Method overview

The baseflow separation workflow includes:

1. selection of the Lyne–Hollick filter parameter (`k`);
2. application of the recursive digital filter;
3. estimation of the baseflow component;
4. estimation of the stormflow component;
5. computation of the Baseflow Index (BFI).

The workflow can directly use the recession parameter estimated from the MRC analysis.

The MRC analysis is performed on the daily streamflow series, while the Lyne–Hollick filter is applied to the original streamflow series.

## Import

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import hydroevents

from hydroevents import (
    compute_mrc,
    separate_baseflow,
)

## Load input data
This notebook requires two outputs from the preprocessing step:

- cleaned hourly streamflow series;
- daily mean streamflow series.

The daily series is used to estimate the recession parameter through the MRC analysis, whereas the hourly series is used for baseflow separation.


In [ ]:
DATA_DIR = Path("../examples/data")

q_file = DATA_DIR / "Q_example.xlsx"

q_day = DATA_DIR / "Q_example_daily.xlsx"

df_q = pd.read_excel(q_file)
df_q_daily = pd.read_excel(q_day)

df_q["Date"] = pd.to_datetime(df_q["Date"])
df_q_daily["Date"] = pd.to_datetime(df_q_daily["Date"])


## Compute recession parameter

The Lyne–Hollick filter parameter can be estimated from the Master Recession Curve workflow.

In [ ]:
df_daily = df_q_daily.copy()

mrc_results = compute_mrc(df_daily)

## Define baseflow separation parameters

The workflow uses the following parameters:

- `K_METHOD`: method used to select the Lyne–Hollick recession parameter from the MRC workflow. This parameter is only used when MRC results are provided;
- `FILL_NAN_WITH_ZERO`: if `True`, missing discharge values are temporarily replaced by zero before filtering. This prevents interruptions of the recursive filter when residual gaps are present. Users should be aware that long missing periods may influence the resulting baseflow separation.
- `DIRECTION`: direction of the Lyne–Hollick filtering pass.
  Available options include:
  - `"f"`: forward filtering
  - `"r"`: reverse filtering
  - combinations such as `"frf"` for multi-pass filtering

Available `K_METHOD` options are:

- `"mrc"`: uses the recession parameter derived from the final cumulative Master Recession Curve fit;
- `"mean"`: uses the mean recession coefficient computed from valid recession segments;
- `"median"`: uses the median recession coefficient computed from valid recession segments.

The selected recession coefficient is automatically converted to the corresponding hourly Lyne–Hollick filter parameter (`k_hour`).

Alternatively, users may bypass the MRC-derived estimates and directly provide a manual value of the hourly Lyne–Hollick filter parameter (*k_hour*). In this case, the selected value is used for the baseflow separation without requiring MRC results.

In [ ]:
K_METHOD = "mrc"
FILL_NAN_WITH_ZERO = True
DIRECTION = "f"

## Apply Lyne–Hollick filter

In [ ]:
bf_results = separate_baseflow(
    df_q,
    q_col="Q",
    date_col="Date",
    mrc_results=mrc_results,
    k_method=K_METHOD,
    nan_to_zero=FILL_NAN_WITH_ZERO,
    direction=DIRECTION
)

bf_results.keys()

## Baseflow Index

The Baseflow Index (BFI) is computed as the ratio between cumulative baseflow and cumulative streamflow.

In [ ]:
bf_results["bfi"]

## Plot baseflow separation

In [ ]:
df_bf = bf_results["df"]

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=df_bf.index,
        y=df_bf["Q_input"],
        mode="lines",
        name="Streamflow",
    )
)

fig.add_trace(
    go.Scatter(
        x=df_bf.index,
        y=df_bf["Baseflow"],
        mode="lines",
        name="Baseflow",
    )
)

fig.add_trace(
    go.Scatter(
        x=df_bf.index,
        y=df_bf["Stormflow"],
        mode="lines",
        name="Stormflow",
    )
)

fig.update_layout(
    title="Baseflow separation using the Lyne–Hollick filter",
    xaxis_title="Date",
    yaxis_title="Q",
    template="plotly_white",
)

fig.show()

## Save output
The output dataframe includes:

- `Q_original`: original streamflow values;
- `Q_input`: streamflow values actually used by the filter;
- `Baseflow`: estimated baseflow component;
- `Stormflow`: estimated quick-flow/stormflow component.

The notebook also reports the Baseflow Index (BFI), the selected filter parameter, and the filtering direction.

In [ ]:
output_dir = Path("../examples/output")

output_dir.mkdir(parents=True, exist_ok=True)

df_bf.to_excel(
    output_dir / "baseflow_separation.xlsx"
)

print(f"Results saved to: {output_dir.resolve()}")